# Advanced document indexing

# Splitting and ingesting HTML content

## Splitting and ingesting the content of a single URL (on Cornwall)

In [1]:
# Run this cell in Google Colab before running the rest of the notebook.
%pip install -q langchain==1.0.3 langchain-openai==1.0.1 langchain-community==0.4.1 langchain-chroma==1.0.0 langchain-openrouter chromadb==1.3.0 lxml==5.4.0 html2text==2025.4.15 lark==1.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB

### Preparing the Chroma DB collections

In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import os
import getpass

try:
    from google.colab import userdata
except ImportError:
    userdata = None

OPENROUTER_API_KEY = None
if userdata is not None:
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass.getpass("Enter your OPENROUTER_API_KEY: ")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ.setdefault("USER_AGENT", "building-llm-applications/ch08-colab")

embedding_model = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

In [4]:
corwnall_granular_collection = Chroma( #A
    collection_name="cornwall_granular",
    embedding_function=embedding_model,
)

corwnall_granular_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists

In [5]:
corwnall_coarse_collection = Chroma( #A
    collection_name="cornwall_coarse",
    embedding_function=embedding_model,
)

corwnall_coarse_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists

### Loading the HTML content with the AsyncHtmlLoader

In [89]:
from langchain_community.document_loaders import AsyncHtmlLoader

In [90]:
# destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
destination_url = "https://en.wikivoyage.org/api/rest_v1/page/html/Cornwall"

In [91]:
# html_loader = AsyncHtmlLoader(destination_url)


In [92]:
# docs = html_loader.load()

In [93]:
import requests
from urllib.parse import quote
from langchain_core.documents import Document

page_title = "Cornwall"

url = f"https://en.wikivoyage.org/api/rest_v1/page/html/{quote(page_title)}"

headers = {
    "User-Agent": "building-llm-applications-ch08/1.0 student-notebook",
    "Accept-Encoding": "gzip",
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

docs = [
    Document(
        page_content=response.text,
        metadata={"source": url, "title": page_title},
    )
]

In [94]:
# from urllib.parse import quote
# from langchain_core.documents import Document

# page_title = "Cornwall"

# url = f"https://en.wikivoyage.org/api/rest_v1/page/html/{quote(page_title)}"

# headers = {
#     "User-Agent": "building-llm-applications-ch08/1.0 student-notebook",
#     "Accept-Encoding": "gzip",
# }

# html_loader = AsyncHtmlLoader(
#     url,
#     header_template=headers,
#     requests_per_second=1,
#     raise_for_status=True,
# )

# docs = html_loader.load()
# for doc in docs:
#     doc.metadata.update({"source": url, "title": page_title})

In [95]:
len(docs)

1

### Splitting into granular chunks with the HTMLSectionSplitter

In [96]:
from langchain_text_splitters import HTMLSectionSplitter

In [97]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on)

In [98]:
def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #A
        temp_chunks = html_section_splitter.split_text(
            html_string) #B
        all_chunks.extend(temp_chunks)

    return all_chunks

#A Extract the HTML text from the document
#B Each chunk is a H1 or H2 HTML section

In [99]:
granular_chunks = split_docs_into_granular_chunks(docs)

#### Ingesting granular chunks

In [100]:
corwnall_granular_collection.add_documents(documents=granular_chunks)

['38f4bc11-d43b-4472-b737-4ff302165511',
 '5bd36c01-a6af-4202-aa1b-5f3b39fb0feb',
 '4c510e28-56cd-4049-9b9c-08ea593e3d1c',
 'f6af2352-69ee-45f7-8d80-4fe63c1152ae',
 '8c9e020c-6425-4a7b-a74a-594da64a859f',
 'acac1e19-8c80-4429-b0f0-c8c3bfd4ec97',
 'b896332e-9a09-438f-bc7b-b35656c46f4e',
 '40ce52fa-8d4d-4cdb-865a-08afe257af89',
 'bd34fde3-0e5c-46c5-82fe-60bfb1f281f8',
 '2299e7ac-6497-4a1f-bf39-d3540da819fd',
 '4f63cdf5-32fe-4af4-9c14-5421bfa72dd5',
 'b9302c46-701c-4a36-a00c-34e2b591f671',
 'becb4f83-9504-41fc-b441-5bb7f8ed410b',
 '383dcbeb-322b-43d1-87b2-7f17f2740d61',
 'bfb05102-daef-4a6a-ac8c-94838db34f60']

#### Searching granular chunks

In [101]:
results = corwnall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

page_content='Festivals 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance of associated rituals. Some towns have a street-parade during this peri

### Splitting into coarse chunks with the RecursiveCharacterTextSplitter

In [102]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [103]:
html2text_transformer = Html2TextTransformer()

In [104]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, chunk_overlap=300
)

In [105]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #A
    coarse_chunks = text_splitter.split_documents(
        text_docs)

    return coarse_chunks
#A transform HTML docs into clean text docs

In [106]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

#### Ingesting coarse chunks

In [107]:
corwnall_coarse_collection.add_documents(documents=coarse_chunks)

['277ad1d6-e5aa-4ea2-8e3e-4132ca50446b',
 '38c836d7-2344-4c7e-82e2-f71e2c51f1ec',
 'f477b90e-1cad-4acb-8b81-f764bc4c100a',
 'f0baeb60-7c5b-46be-9935-501893f1084e',
 'eb278c27-6fce-40c5-ad50-8a0f380b3526',
 '3be55895-8f03-4591-b530-ebb5aea41bd3',
 '341f9c7e-2834-4c98-b69a-b2e754a608fe',
 'd72e118c-b4c6-448a-bccb-0a4cc6f5e996',
 'def73f9f-9aeb-4bbd-b52c-ae737adc6399',
 '85562a27-1c47-443f-9fde-72c026d9987c',
 '4c3da50a-1dc0-4b84-ba16-cb37fa05a79d',
 '7381a9ca-db62-418f-8f58-d93e651ae6e4',
 '24395cc4-487e-4244-b9a1-656126b0b16c']

#### Searching coarse chunks

In [108]:
results = corwnall_coarse_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

page_content='_See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Cornish fishing communities. The celeb

## Splitting and ingesting the content of various URLs (across UK destinations)

### Preparing the Chroma DB collections

In [109]:
uk_granular_collection = Chroma( #A
    collection_name="uk_granular",
    embedding_function=embedding_model,
)

uk_granular_collection.reset_collection() #B

In [110]:
uk_coarse_collection = Chroma( #A
    collection_name="uk_coarse",
    embedding_function=embedding_model,
)

uk_coarse_collection.reset_collection() #B

### Splitting and ingesting HTML content with the HTMLSectionSplitter

In [112]:
# Reduce this list if you want to save on processing fees
# uk_destinations = [
#     "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
#     "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
#     "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven"
#     "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
#     "Rye_(England)", "Seaford", "Ashdown_Forest"
# ]

uk_destinations = [
     "Cornwall", "East_Sussex", "Polperro", "Ashdown_Forest"
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

wikivoyage_headers = {
    "User-Agent": "building-llm-applications-ch08/1.0 student-notebook",
    "Accept-Encoding": "gzip",
}

def load_wikivoyage_html(url):
    try:
        response = requests.get(url, headers=wikivoyage_headers, timeout=30)
        response.raise_for_status()
        return [Document(page_content=response.text, metadata={"source": url})]
    except Exception as e:
        print(f"Skipping {url}: {e}")
        return []

In [113]:
# import asyncio
# uk_destinations = [
#      "Cornwall", "East_Sussex", "Polperro", "Ashdown_Forest"
# ]

# wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

# wikivoyage_headers = {
#         "User-Agent": "building-llm-applications-ch08/1.0 student-notebook",
#         "Accept-Encoding": "gzip",
# }

# def load_wikivoyage_html(url):
#     try:
#         html_loader = AsyncHtmlLoader(
#             url,
#             header_template=wikivoyage_headers,
#             requests_per_second=0.2,
#             raise_for_status=True,
#             ignore_load_errors=False,
#         )
#         return html_loader.load()
#     except Exception as e:
#         print(f"Skipping {url}: {e}")
#         return []

In [114]:
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]

In [115]:
print(uk_destination_urls)

['https://en.wikivoyage.org/wiki/Cornwall', 'https://en.wikivoyage.org/wiki/East_Sussex', 'https://en.wikivoyage.org/wiki/Polperro', 'https://en.wikivoyage.org/wiki/Ashdown_Forest']


In [88]:
# for destination_url in uk_destination_urls:
#     html_loader = AsyncHtmlLoader(destination_url) #C
#     docs =  html_loader.load() #D

#     for doc in docs:
#         print(doc.metadata)
#         granular_chunks = split_docs_into_granular_chunks(docs)
#         uk_granular_collection.add_documents(documents=granular_chunks)

#         coarse_chunks = split_docs_into_coarse_chunks(docs)
#         uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists
#C Loader for one destination
#D Documents of one destination

In [116]:
for destination_url in uk_destination_urls:
    docs = load_wikivoyage_html(destination_url) #C
    if not docs:
        continue

    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists
#C Loader for one destination
#D Documents of one destination

{'source': 'https://en.wikivoyage.org/wiki/Cornwall'}
{'source': 'https://en.wikivoyage.org/wiki/East_Sussex'}
{'source': 'https://en.wikivoyage.org/wiki/Polperro'}
{'source': 'https://en.wikivoyage.org/wiki/Ashdown_Forest'}


#### Searching

In [117]:
granular_results = uk_granular_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in granular_results:
    print(doc)

page_content='East Sussex' metadata={'Header 1': 'East Sussex'}
page_content='Sussex for free 
 [ edit ] 
 
 A Market during the Brighton Festival 
 There's plenty in Sussex for those who don't wish to spend plenty of cash on attractions: 
 
 
 Walking  - 3,500   km of walking paths, bridleways, scenic roads - all for free. 
 Go for a swim: Sussex has some of the cleanest beaches in the UK, with Brighton Beach renowned for its packed seafront, less well used areas, such as Eastbourne, Bexhill and Hastings still have facilities and cleanliness. 
 Brighton itself can be one big performance, the  Brighton Festival  and the  Brighton Festival Fringe , Features street performers, theatre groups, musicians, guided walks and a whole host of other great activities. 
 Town museums: Often they will charge, but some such as  Brighton Museum and Art Gallery  and Newhaven Museum are free (donations are gratefully welcomed though).' metadata={'Header 2': 'Sussex for free'}
page_content='Towns and vi

In [33]:
print(granular_results)

[Document(id='c9d7936d-c01d-4f0e-925f-b418bba6d44e', metadata={'Header 1': 'East Sussex'}, page_content='East Sussex'), Document(id='815816f0-40f9-4450-b1fc-961b586d76f4', metadata={'Header 2': 'Sussex for free'}, page_content="Sussex for free \n [ edit ] \n \n A Market during the Brighton Festival \n There's plenty in Sussex for those who don't wish to spend plenty of cash on attractions: \n \n \n Walking  - 3,500 \xa0 km of walking paths, bridleways, scenic roads - all for free. \n Go for a swim: Sussex has some of the cleanest beaches in the UK, with Brighton Beach renowned for its packed seafront, less well used areas, such as Eastbourne, Bexhill and Hastings still have facilities and cleanliness. \n Brighton itself can be one big performance, the  Brighton Festival  and the  Brighton Festival Fringe , Features street performers, theatre groups, musicians, guided walks and a whole host of other great activities. \n Town museums: Often they will charge, but some such as  Brighton Mu

In [34]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Events or festivals in Polperro",k=4)
for doc in coarse_results:
    print(doc)

page_content='[edit]

50°19′52″N 4°31′11″W

Map of Polperro

The walk into town from the parking lot is not very steep and takes 10
minutes. If walking is not your thing, there's a horse and cart or converted
milk float "tram" that will take you there and back for £1.50 (75p one way).

## See

[edit]

There are a couple of art galleries on the walk from the car park to the
harbour that may be worth a visit.The coastpath to the nearby town of Looe
also makes a pleasant walk on a summer's day.

  * 50.331685-4.5173881 Polperro Harbour Heritage Museum, 4 The Warren, PL13 2RB, ☏ +44 1503 272423. 10.30AM-4.30PM. £3 (adult). (updated Jul 2022)
  * 50.331648-4.5213152 Polperro Model Village, Mill Hill, PL13 2RP, ☏ +44 1503 272378. Miniature representation of the village, model railway and museum of local myths and legends. (updated Jul 2022)

## Do

[edit]

**Sea trips** from the harbour. There have been plenty of sightings of basking
sharks just off Polperro Harbour and the converted fishing

In [118]:
granular_results = uk_granular_collection.similarity_search(
    query="Beaches in Conrwall",k=4)
for doc in granular_results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='Regions 
 [ edit ] 
 
 .mw-parser-output .wv-staticMap{position:relative;left:-3px;margin-top:3px} 
 50°19′41″N 5°1′12″W Map of Cornwall 
 
 
Wikivoyage divides Cornwall into three regions. The  Isles of Scilly  are covered in a separate article. 
 
 .mw-parser-output .regionlistitem-table{vertical-align:middle;border-collapse:separate;border-spacing:2px;margin:0}.mw-parser-output .regionlistitem-table td{padding:0.15em 0.4em}.mw-parser-output .regionlistitem-textholder{transition:background-color 0.18s ease;border-radius:4px;cursor:pointer}.mw-parser-output .regionlistitem-textholder:hover{background-color:rgba(0,0,0,0.06);color:inherit}.mw-parser-output .regionlistitem-table:hover .regionlistitem-colorcell{background:inherit;color:inherit}@media screen{html.skin-theme-clientpref-night .mw-parser-output .regionlistitem-textholder:hover{background-color:rgba(255,255,255,0.06);color:inherit}}@media screen and (prefe

In [119]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Beaches in Cornwall",k=4)
for doc in coarse_results:
    print(doc)

page_content='_For other places with the same name, seeCornwall (disambiguation)._

**Cornwall** (Cornish: _Kernow_) is a county in the southwest of England.
Lying west of Devon from which it is separated by the River Tamar, Cornwall is
one of the more isolated and distinctive parts of the United Kingdom but is
also one of its most popular with holidaymakers. Its relatively warm climate,
long coastline, amazing scenery, and diverse Celtic heritage (combined with
tales of smuggling, pirates and King Arthur!) go only part of the way to
explaining its appeal.

The biomes that house the Eden Project, near St. Austell, Mid-Cornwall.

Cornwall is a popular destination for those interested in cultural tourism,
due to its long association with visual and written arts and its wealth of
archaeology. Its mining heritage has been recognised by the United Nations
(UNESCO). Over 30% of the county is designated as an Area of Outstanding
Natural Beauty (AONB), giving it national status and protection.

# Embedding strategy

## Embedding child chunks with ParentDocumentRetriever

In [120]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Setting up the Parent Document retriever

In [121]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=embedding_model,
)

child_chunks_collection.reset_collection() #D

doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

In [123]:
for destination_url in uk_destination_urls:
    html_docs = load_wikivoyage_html(destination_url) #A
    # html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D

#A Loader for destination web page
#B HTML documents of one destination
#C Transform HTML docs into clean text deocs
#D Ingest coarse chunks into document store and granular chunks into vector store

Ingesting https://en.wikivoyage.org/wiki/Cornwall
Ingesting https://en.wikivoyage.org/wiki/East_Sussex
Ingesting https://en.wikivoyage.org/wiki/Polperro
Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [124]:
list(doc_store.yield_keys())
#A Show the keys of the added coarse chunks

['22843eb6-5aa5-46a2-81fc-293d014960b3',
 '5245b5bd-6cec-4dfd-b044-92475e7a7b59',
 '5de6c2dc-c83b-46a6-879a-585925b99489',
 'de205f01-85ff-438e-9b11-6f5afc5875dc',
 'c2921afa-0694-4eb0-ab07-e1c6a16afa9e',
 '75d8bae6-0ff9-4715-8ada-9cd50b90d135',
 '1689c5e1-c43c-4c49-9593-6d86a9920e32',
 '541e9b91-f94e-4a21-8272-d3dc8114d014',
 'c44997db-c621-490a-9074-1274f63d9e34',
 '1abc7b23-ff74-454e-be82-347e33ee5e91',
 '9e87f33a-5108-430e-9876-4f0db561d0a9',
 'a664131a-ea3e-4c4e-aa9d-d8f816b99cff',
 'af146d14-7044-4175-80c0-3f1393a69d55',
 '1579518a-1424-49d9-8a68-d14d2f9137bd',
 'c3805f07-91a2-4529-ac9f-b7ce575202b6',
 '9277a7a9-23c4-45b4-a2f2-f1c75a3a560e',
 '097e6c2c-6d69-4b5c-92d4-d8bd9f86ffff',
 'c51f0880-3266-4155-8c2a-6770b76a9028',
 '5957e757-a1db-4db5-a2d7-1817bc0f0121',
 '82bedae1-5965-4bd4-bb03-bdfbe0d3ce86',
 'da42dc4c-2844-4bfc-9629-5b180ea6b961',
 '95273571-5ee2-4a0c-8696-f5a3a1798087',
 'e874d1a0-4b0d-4c44-ae17-e698fb6a3b10',
 '9c922bbb-1bda-4fc9-b2bc-3cc4b7fadffb',
 '07de0474-2ce9-

### Performing a search on granular information

In [125]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

In [126]:
len(retrieved_docs)

3

In [127]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content="The Cornish have many significant saints. The pre-eminent patron saint is\nSaint Piran, whose flag, black with a white cross, is widely regarded as the\nnational emblem of Cornwall and can be seen all across the county. It is flown\nfrom private homes and government and public buildings. Saint Piran's Day is\nwidely celebrated on March 5 in Cornwall and amongst the Cornish diaspora\naround the globe.\n\nCornwall was a contributor to the Industrial Revolution, being famous\nparticularly for its copper and tin mining. Cornish miners have emigrated to\nmany parts of the world to the extent that the Cornish claim that a mine is\ndefined as being a hole in the ground with a Cornishman at its bottom. The\nCornish mines pioneered the use of stationary steam engines to power the\nmines. The Cornish are extremely proud of their history and heritage, which\npre-date the arrival of the Romans or Anglo-Saxons in

In [128]:
retrieved_docs[1]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content='### Cornish\n\n[edit]\n\nBilingual welcome sign in Penzance station The Cornish word "tre" A common\nCornish word used in placenames is "tre" which refers to a farmstead, home,\nhomestead, town or village depending on context. Many towns and villages start\nwith this word such as: Trekenner, Treknow and Treburley  \n---  \n  \n**Cornish** (_Kernowek/Kernewek_) is a language belonging to the Brythonic\nbranch of the Celtic languages, and closely related to Breton and Welsh. It\nwas traditionally the dominant language of Cornwall, though the number of\nspeakers had diminished by the 17th century, and it became extinct some time\nlater. It is claimed that the last speaker was Dolly Pentreath, a fishwife\nfrom Mousehole, who passed away in Mousehole on 26 December 1777, although\nothers claim that there were Cornish-speakers who lived into the early 20th\ncentury.\n\nCornish was revived in the early 20th

### Comparing with direct semantic search on child chunks

In [129]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [130]:
len(child_docs_only)

4

In [131]:
child_docs_only[0]

Document(id='70a463c7-59d4-4b3f-a569-9523239f6af7', metadata={'doc_id': 'de205f01-85ff-438e-9b11-6f5afc5875dc', 'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content='### Cornish\n\n[edit]')

In [132]:
# IMPORTANT: as you can see a granular search would have identified the chunk, but it would have lost the usefulcontext about travelling in Cornwall

## Embedding child chunks with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(
        openai_api_key=OPENAI_API_KEY),
)

child_chunks_collection.reset_collection() #D

doc_byte_store = InMemoryByteStore() #E
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #F
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E

        coarse_chunk_id = coarse_chunks_ids[i]

        granular_chunks = child_splitter.split_documents(
            [coarse_chunk]) #F

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id #G

        all_granular_chunks.extend(granular_chunks)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_granular_chunks) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into parent coarse chunks
#E Iterate over the parent coarse chunks
#F Create child granular chunks form each parent coarse chunk
#G Link each child granular chunk to its parent coarse chunk
#H Ingest the child granular chunks into the vector store
#I Ingest the parent coarse chunks into the document store

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 12.74it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 14.46it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 16.68it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 14.20it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 14.45it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke(
    "Cornwall Ranger")

In [ ]:
len(retrieved_docs)

4

In [ ]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="Cornwall boasts many attractions for the traveller, many lying outside of\ncities and towns amidst the Cornish landscape:\n\n  * Within the 208 m² of the Bodmin Moor, is **King Arthur's Hall** , a megalithic monument and **Brown Willy** , the highest point in Cornwall at 417 m (1,368 ft). **Dozmary Pool** is a small beautiful lake where, according to legend, King Arthur was entrusted with the sword Excalibur. There is also a reputed **Beast of the Moor** , a large wild-cat that haunts and stalks at night, but is similar in fantasy to the Loch Ness Monster, in that no one can prove it exists, though sightings, theories and track-marks abound.\n  * The **Eden Project** , near St Austell, a fabulous collection of flora from all over the planet housed in two 'space age' transparent domes.\n  * The **Lost Gardens of Heligan** \\- near Mev

In [ ]:
##IMPORTANT: same as Parent Document retriever, but more control and flexibility on how to link child to parent chunks

### Comparing with direct semantic search on child chunks

In [ ]:
child_docs_only =  child_chunks_collection.similarity_search(
    "Cornwall Ranger")

In [ ]:
len(child_docs_only)

4

In [ ]:
child_docs_only[0]

Document(id='a18b9a25-f88f-433b-8819-2d80d8a39fcd', metadata={'doc_id': 'f69a9d0a-6153-4c74-9f19-83942aeb3876', 'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'language': 'en', 'title': 'South Cornwall – Travel guide at Wikivoyage'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.')

In [ ]:
## IMPORTANT: Same as before

## Embedding summaries with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

### Setting up the Multi vector retriever (similar to when embedding child chunks)

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A

summaries_collection = Chroma( #B
    collection_name="uk_summaries",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

summaries_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the summarization chain

In [ ]:
llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY)

In [ ]:
summarization_chain = (
    {"document": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{document}") #B
    | llm
    | StrOutputParser())

#A Grab the text content from the document
#B Instantiate a prompt asking to generate summary of the provided text
#C Send the LLM the instantiated prompt
#D Extract the summary text from the response

### Ingesting the coarse chunks and related summaries into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E

        coarse_chunk_id = coarse_chunks_ids[i]

        summary_text =  summarization_chain.invoke(
            coarse_chunk) #F
        summary_doc = Document(page_content=summary_text,
                               metadata={doc_key: coarse_chunk_id})

        all_summaries.append(summary_doc) #G

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_summaries) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a summary for the coarse chunk thorugh the summarization chain
#G Link each summary to its related coarse chunk
#H Ingest the summaries into the vector store
#I Ingest the coarse chunks into the document store

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 11.12it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 18.13it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 18.90it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 17.62it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 11.98it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [ ]:
# COMMENT: the code above is similar to when ingesting child chunks, but it is slower because of the summarization step
# which invokes the LLM.
# The processing can be speeded up by parallelizing the outer for loop on the destination urls.

### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")

In [ ]:
len(retrieved_docs)

4

In [ ]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="### By car\n\n[edit]\n\nCornwall can be accessed by road via the A30 which runs from the end of the M5\nat Exeter, all the way through the heart of Devon and Cornwall down to Land's\nEnd. It is a grade-separated expressway as far as Carland Cross near Truro\n(the expressway is expected to be open as far as Camborne (between Redruth and\nHayle) by March 2024). You can also get to Cornwall via the A38, crossing the\nRiver Tamar at Plymouth via the Tamar Bridge, which levies a toll on eastbound\nvehicles. On summer Saturdays and during bank holiday weekends roads to\nCornwall are usually busy.\n\n### By plane\n\n[edit]\n\n50.440833-4.9952781 Cornwall Airport (**NQY** IATA) in Newquay is the main\nairport for the county, with year-round flights only from Aberdeen, Alicante,\nDublin, London Gatwick, and Manchester. During the

### Comparing with direct semantic search on summaries

In [ ]:
summary_docs_only =  summaries_collection.similarity_search(
    "Cornwall Travel")

In [ ]:
len(summary_docs_only)

4

In [ ]:
summary_docs_only

[Document(id='9553aa2d-456e-4a38-8f95-28f072f13992', metadata={'doc_id': '14554480-abf6-4b35-ae7d-8580a27ecc9a'}, page_content="Cornwall offers a diverse array of attractions spanning natural beauty, legends, gardens, historic sites, arts, and heritage, including both independent sites and National Trust properties.\n\n- Natural and legendary sights: King Arthur's Hall and Brown Willy on Bodmin Moor; Dozmary Pool and tales of the Beast of the Moor.\n- Gardens and nature: The Eden Project’s two glass domes; the Lost Gardens of Heligan near Mevagissey.\n- Castles, archaeology, and coastal culture: Tintagel Castle (Arthurian legends and early medieval finds); Minack Theatre (clifftop outdoor theatre and museum); St Michael's Mount.\n- Arts and museums: Tate St Ives (modern art); National Maritime Museum, Falmouth (small-boat collection and other exhibits).\n- Mining and industrial heritage: Historic tin/copper mine sites such as Geevor Tin Mine, Poldark Mine, King Edward Mine, Crown Hill 

In [ ]:
# COMMENT: a direct search on summaries retrieves denser information, but it is missing out on useful details.
# However, you might consider using the summaries directly if after testing they prove adequate.

## Embedding hypothetical questions with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid
from typing import List
from pydantic import BaseModel, Field

### Setting up the Multi vector retriever (same as when embedding summaries)

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A

hypothetical_questions_collection = Chroma( #B
    collection_name="uk_hypothetical_questions",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

hypothetical_questions_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=hypothetical_questions_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the chain to generate hypothetical questions

In [ ]:
class HypotheticalQuestions(BaseModel):
    """A list of hypotetical questions for given text."""

    questions: List[str] = Field(..., description="List of hypothetical questions for given text")

In [ ]:
llm_with_structured_output = ChatOpenAI(
    model="gpt-5-nano",
    openai_api_key=OPENAI_API_KEY).with_structured_output(
        HypotheticalQuestions
)

In [ ]:
hypothetical_questions_chain = (
    {"document_text": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template( #B
        "Generate a list of exactly 4 hypothetical questions that the below text could be used to answer:\n\n{document_text}"
    )
    | llm_with_structured_output #C
    | (lambda x: x.questions) #D
)

#A Grab the text content from the document
#B Instantiate a prompt asking to generate 4 hypothetical questions on the provided text
#C Invoke the LLM configured to return an object containing the questions as a typed list of strings
#D Grab the list of questions from the response

### Ingesting the coarse chunks and related hypothetical questions into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_hypothetical_questions = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E

        coarse_chunk_id = coarse_chunks_ids[i]

        hypothetical_questions = hypothetical_questions_chain.invoke(
            coarse_chunk) #F
        hypothetical_questions_docs = [Document(
            page_content=question, metadata={doc_key: coarse_chunk_id})
                    for question
                    in hypothetical_questions] #G

        all_hypothetical_questions.extend(hypothetical_questions_docs)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_hypothetical_questions) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a list of hypothetical questions for the coarse chunk thorugh the question generation chain
#G Link each hypothetical question to its related coarse chunk
#H Ingest the hypothetical questions into the vector store
#I Ingest the coarse chunks into the document store

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  9.29it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 12.01it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  2.23it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  9.02it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  9.97it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke(
    "How can you go to Brighton from London?")

In [ ]:
len(retrieved_docs)

4

In [ ]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Brighton', 'title': 'Brighton – Travel guide at Wikivoyage', 'language': 'en'}, page_content='Brighton  \n---  \nClimate chart (explanation)  \n| J| F| M| A| M| J| J| A| S| O| N| D  \n---|---|---|---|---|---|---|---|---|---|---|---  \n88 8 3 |  60 8 3 |  51 9 4 |  58 12 6 |  56 16 9 |  50 18 12 |  54 20 14 |  62 21 14 |  67 18 12 |  105 15 9 |  103 11 6 |  97 9 4  \nAverage max. and min. temperatures in °C  \nPrecipitation+Snow totals in mm  \nSource: Wikipedia. Visit the Met Office for a five day forecast.  \n| Imperial conversion  \n---  \nJ| F| M| A| M| J| J| A| S| O| N| D  \n3.5 46 37 |  2.4 46 37 |  2 48 39 |  2.3 54 43 |  2.2 61 48 |  2 64 54 |  2.1 68 57 |  2.4 70 57 |  2.6 64 54 |  4.1 59 48 |  4.1 52 43 |  3.8 48 39  \nAverage max. and min. temperatures in °F  \nPrecipitation+Snow totals in inches  \n  \nThe city is close to London, and is increasingly popular with media and music\ntypes who don\'t want to live in t

### Inspecting possible questions matching our question through semantic search

In [ ]:
hypothetical_question_docs_only = hypothetical_questions_collection.similarity_search(
    "How can you go to Brighton from London?")

In [ ]:
len(hypothetical_question_docs_only)

4

In [ ]:
hypothetical_question_docs_only

[Document(id='399bb54e-88e8-4eb1-b572-d363e8f23006', metadata={'doc_id': 'a1f51d6c-2d27-4a6c-b1a0-4f9918dc1545'}, page_content='How can you travel to Brighton by train from London, and what are the two main railway stations in the city?'),
 Document(id='65d2f229-e7e5-4516-ac09-ea93fd3feb1d', metadata={'doc_id': '53e1031c-3d92-4e7e-9398-1770b215dad1'}, page_content='What transportation options are described for getting to Brighton and for getting around the city?'),
 Document(id='fb27db61-b09a-4280-9473-cfe82759e708', metadata={'doc_id': 'f4b555cf-afe0-43e2-b10a-f59fdab3e2e2'}, page_content='What is the fastest way to travel from Gatwick to Brighton, and how long does it take by train according to the text?'),
 Document(id='320ca9aa-f328-4c06-9d18-00ef76577884', metadata={'doc_id': '79b9f77f-21ab-4ab4-bbf4-4a04d5576cbc'}, page_content='If I want to travel around Brighton all day on buses with one fare, what ticket would I buy and how much would it cost?')]

# Granular chunk expansion with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [ ]:
granular_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #A

granular_chunks_collection = Chroma( #B
    collection_name="uk_granular_chunks",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
)

granular_chunks_collection.reset_collection() #C

expanded_chunk_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=granular_chunks_collection,
    byte_store=expanded_chunk_store
)
#A Splitter to generate granular chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host expanded chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Ingesting granular and expanded chunks into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    granular_chunks = granular_chunk_splitter.split_documents(
        text_docs) #D

    expanded_chunk_store_items = []
    for i, granular_chunk in enumerate(
        granular_chunks): #E

        this_chunk_num = i #F
        previous_chunk_num = i-1 #F
        next_chunk_num = i+1 #F

        if i==0: #F
            previous_chunk_num = None
        elif i==(len(granular_chunks)-1): #F
            next_chunk_num = None

        expanded_chunk_text = "" #G
        if previous_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                previous_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_text += granular_chunks[
            this_chunk_num].page_content #G
        expanded_chunk_text += "\n"

        if next_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                next_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_id = str(uuid.uuid4()) #H
        expanded_chunk_doc = Document(
            page_content=expanded_chunk_text) #I

        expanded_chunk_store_item = (expanded_chunk_id,
                                     expanded_chunk_doc)
        expanded_chunk_store_items.append(
            expanded_chunk_store_item)

        granular_chunk.metadata[
            doc_key] = expanded_chunk_id #J

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        granular_chunks) #K
    multi_vector_retriever.docstore.mset(
        expanded_chunk_store_items) #L

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into granular chunks
#E Iterate over the granular chunks
#F determine the index of the current chunk and its previous and next chunks
#G Assemble the text of the expanded chunk by including the previous and next chunk
#H Generate the ID of the expanded chunk
#I Create the expanded chunk document
#J Link each granular chunk to its related expanded chunk
#K Ingest the granular chunks into the vector store
#L Ingest the expanded chunks into the document store

Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00,  9.39it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 12.60it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 14.08it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 13.77it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|####################################################################| 1/1 [00:00<00:00, 13.99it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")

In [ ]:
len(retrieved_docs)

4

In [ ]:
retrieved_docs[0]

Document(metadata={}, page_content="Buses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\nserving a number of other towns on branch lines. For train times and fares\nvisit National Rail Enquiries.\nThe **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.\n\nThe **Lo

### Comparing with direct semantic search on granular chunks

In [ ]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [ ]:
len(child_docs_only)

4

In [ ]:
child_docs_only[0]

Document(id='a18b9a25-f88f-433b-8819-2d80d8a39fcd', metadata={'language': 'en', 'doc_id': 'f69a9d0a-6153-4c74-9f19-83942aeb3876', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'source': 'https://en.wikivoyage.org/wiki/South_Cornwall'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.')

In [ ]:
# COMMENT: the expanded chunk has more useful context